In [1]:
from moe_reft import interventions_config, tiny_sft, read_config, datamodels, sft_dataset
from moe_reft.olmoe import modeling_olmoe, configuration_olmoe, load_weights
import torch
from transformers import AutoTokenizer
from loguru import logger

config_path = "moe_reft/configs/olmoe.yaml"
model_name: str = "allenai/OLMoE-1B-7B-0125"
tokenizer_model_name: str = "allenai/OLMoE-1B-7B-0125-Instruct"

train_config, interventions_config_, _ = read_config.load_all_configs(config_path)

custom_model = modeling_olmoe.OlmoeForCausalLM(
    configuration_olmoe.OlmoeInterventionsConfig(interventions_config=interventions_config_)
)

report = load_weights.load_hf_into_custom_model(
    hf_model_name_or_path=model_name,
    custom_model=custom_model,
    intervention_patterns=["*.pre_moe_intervention.*", "*.after_moe_intervention.*"],
    map_dtype=torch.bfloat16,  # optional casting
    map_device=torch.device("cuda"),  # optional device move
    trust_remote_code=False,
)
logger.info(f"{report.summary()}")

for name, param in custom_model.named_parameters():
    if load_weights.matches_any(name, interventions_config.INTERVENTION_PATTERNS):
        param.requires_grad = True

# 5) Print parameter stats
total_params, trainable_params = load_weights._count_parameters(custom_model)
print(f"Total parameters:     {total_params}")
print(f"Trainable parameters: {trainable_params}")

logger.info(f"Parameter stats — total: {total_params}, trainable: {trainable_params}")
# dataloader, _, dataset = tiny_sft.build_tiny_sft_dataloader(model_name=tokenizer_model_name)
# train_sft(model=custom_model, dataloader=dataloader, train_config=train_config)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
response_template = sft_dataset.extract_response_template(tokenizer)
response_template_ids = tokenizer(response_template)["input_ids"]

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Building partial state dict: 100%|██████████| 3219/3219 [00:06<00:00, 533.32it/s]
2025-11-26 12:14:12.676 | INFO     | moe_reft.olmoe.load_weights:load_hf_into_custom_model:171 - Parameter stats — total: 6986287104, trainable: 67125248
2025-11-26 12:14:12.791 | INFO     | __main__:<module>:25 - Copied: 3219 | Skipped (shape): 0 | Skipped (missing): 96 | Skipped (intervention): 0
2025-11-26 12:14:12.824 | INFO     | __main__:<module>:36 - Parameter stats — total: 6986287104, trainable: 67125248


Total parameters:     6986287104
Trainable parameters: 67125248
Total parameters:     6986287104
Trainable parameters: 67125248


In [7]:
for name, param in custom_model.named_parameters():
    if load_weights.matches_any(name, interventions_config.INTERVENTION_PATTERNS):
        param.requires_grad = True

In [1]:
from moe_reft import sft_dataset

tokenizer_model = "allenai/OLMoE-1B-7B-0125-Instruct"
ds = sft_dataset.SFTDataset(
    source="openai/gsm8k",
    tokenizer_model_name=tokenizer_model,
    system_key=None,
    system_message="You are a helpful math tutor. Solve step by step.",
    user_key="question",
    assistant_key="answer",
    split="train",
    name="main",
)


/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-11-13 01:03:18.181 | INFO     | moe_reft.sft_dataset:__init__:49 - For the tokenizer_model_name='allenai/OLMoE-1B-7B-0125-Instruct' automatically assigned the response template to 
<|assistant|>

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
2025-11-13 01:03:20.479 | INFO     | moe_reft.sft_dataset:validate_one_sample:73 - Decoded Input (Full Prompt + Response)
|||IP_ADDRESS|||<|system|>
You are a helpful math tutor. Solve step by step.
<|user|>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
<|assistant|>
Natalia sold 48/2 = <<48/2=24>>24 clips in 

In [2]:
for sample in ds:
    input_ids, labels = sample["input_ids"], sample["labels"]
    break
    

AssertionError: Length mismatch: system=155, user=155, assistant=126

In [6]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k",split="train",name="main")

In [7]:
ds[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}

In [1]:
from moe_reft import sft_dataset
from transformers import AutoTokenizer


tokenizer_model_name = "allenai/OLMoE-1B-7B-0125-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
train_dataset = sft_dataset.SFTDataset(
        source="openai/gsm8k",
        tokenizer=tokenizer,
        system_key=None,
        system_message="You are a helpful math tutor. Solve step by step.",
        user_key="question",
        assistant_key="answer",
        split="train",
        name="main",
    )
val_dataset = sft_dataset.SFTDataset(
    source="openai/gsm8k",
    tokenizer=tokenizer,
    system_key=None,
    system_message="You are a helpful math tutor. Solve step by step.",
    user_key="question",
    assistant_key="answer",
    split="test",
    name="main",
)

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
2025-11-16 19:15:29.880 | INFO     | moe_reft.sft_dataset:__init__:100 - For the tokenizer.name_or_path='allenai/OLMoE-1B-7B-0125-Instruct' automatically assigned the response template to 
<|assistant|>

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
2025-11-16 19:15:31.710 | INFO     | moe_reft.sft_dataset:validate_one_sample:124 - Decoded Input (Full Prompt + Response)
|||IP_ADDRESS|||<|system|>
You are a helpful math tutor. Solve step by step.
<|user|>
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
<|assistant|>
Natalia sold 48/2 = <<48/2=24>>24 clips

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import torch
from transformers import PreTrainedTokenizerBase

# Whatever you already use in SFTTransform
CROSS_ENTROPY_IGNORE_INDEX = -100  # or import from your constants


@dataclass
class SFTDataCollator:
    tokenizer: PreTrainedTokenizerBase
    label_pad_token_id: int = CROSS_ENTROPY_IGNORE_INDEX
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # 1. Separate labels so tokenizer.pad only sees model inputs
        labels_list: List[Any] = [f["labels"] for f in features]
        features_for_pad: List[Dict[str, Any]] = [
            {k: v for k, v in f.items() if k != "labels"} for f in features
        ]

        # 2. Let tokenizer.pad handle input_ids / attention_mask
        batch = self.tokenizer.pad(
            features_for_pad,
            padding=True,                 # pad to max length in this batch
            max_length=None,              # or a fixed max_length if you want
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        # 3. Manually pad labels to match seq_len of input_ids
        seq_len: int = batch["input_ids"].size(1)
        padded_labels: List[List[int]] = []

        for lbl in labels_list:
            # convert to python list of ints
            if isinstance(lbl, torch.Tensor):
                lbl_list = lbl.tolist()
            else:
                lbl_list = list(lbl)

            # truncate if somehow longer than seq_len
            if len(lbl_list) > seq_len:
                lbl_list = lbl_list[:seq_len]

            pad_len = seq_len - len(lbl_list)
            if pad_len > 0:
                lbl_list = lbl_list + [self.label_pad_token_id] * pad_len

            padded_labels.append(lbl_list)

        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)

        return batch


In [3]:
from torch.utils.data import DataLoader

collator = SFTDataCollator(tokenizer=train_dataset.tokenizer)

train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=1,
        pin_memory=True,
        collate_fn = collator
    )

In [10]:
# for td in train_loader:
#     print(td['input_ids'].shape)
    # break

In [5]:
td['input_ids'].shape

torch.Size([32, 376])

In [ ]:
from moe_reft import sft_dataset

tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name)
response_template = sft_dataset.extract_response_template(tokenizer)

train_dataset = sft_dataset.SFTDataset(
        source="openai/gsm8k",
        tokenizer=tokenizer,
        response_template_ids=tokenizer(response_template)['input_ids'],
        system_key=None,
        system_message="You are a helpful math tutor. Solve step by step.",
        user_key="question",
        assistant_key="answer",
        split="train",
        name="main",
)

/home/recoverx/astarag/MoE-ReFT/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
tokenizer(response_template)['input_ids']

[187, 29, 93, 515, 5567, 49651, 187]

In [3]:
from moe_reft.olmoe import modeling_olmoe
import torch

model = torch.load("gsm8k_run2/checkpoint-epoch0.pt", weights_only=True)

In [10]:
from moe_reft.olmoe import configuration_olmoe, modeling_olmoe
from moe_reft import read_config

state_dict = model['model_state_dict']
config_path = "moe_reft/configs/olmoe.yaml"
train_config, interventions_config_, _ = read_config.load_all_configs(config_path)

hf_model = modeling_olmoe.OlmoeForCausalLM(
    configuration_olmoe.OlmoeInterventionsConfig(interventions_config=interventions_config_)
)
hf_model.load_state_dict(state_dict)

RuntimeError: Error(s) in loading state_dict for OlmoeForCausalLM:
	Missing key(s) in state_dict: "model.layers.0.pre_moe_intervention.embed_dim", "model.layers.0.pre_moe_intervention.interchange_dim", "model.layers.0.pre_moe_intervention.rotate_layer.parametrizations.weight.original", "model.layers.0.pre_moe_intervention.rotate_layer.parametrizations.weight.0.base", "model.layers.0.pre_moe_intervention.learned_source.weight", "model.layers.0.pre_moe_intervention.learned_source.bias", "model.layers.1.pre_moe_intervention.embed_dim", "model.layers.1.pre_moe_intervention.interchange_dim", "model.layers.1.pre_moe_intervention.rotate_layer.parametrizations.weight.original", "model.layers.1.pre_moe_intervention.rotate_layer.parametrizations.weight.0.base", "model.layers.1.pre_moe_intervention.learned_source.weight", "model.layers.1.pre_moe_intervention.learned_source.bias", "model.layers.2.pre_moe_intervention.embed_dim", "model.layers.2.pre_moe_intervention.interchange_dim", "model.layers.

In [ ]:
from moe_reft.load_weights import load_hf_into_custom_model

report = load_hf_into_custom_model(
        # hf_model_name_or_path="allenai/OLMoE-1B-7B-0125-Instruct",
        pt_file="gsm8k_run2/checkpoint-epoch0.pt",
        custom_model=custom_model,
        intervention_patterns=["*.pre_moe_intervention.*", "*.after_moe_intervention.*"],
        map_dtype=torch.float32,  # optional casting
        map_device=torch.device("cuda"),  # optional device move
        trust_remote_code=False,
    )